In [ ]:



import os, glob, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
import holidays

# ======================
# Paths
# ======================
BASE         = r"C:\Users\User\Desktop\lg aimers"
TRAIN_PATH   = os.path.join(BASE, "train", "train.csv")
SAMPLE_PATH  = os.path.join(BASE, "sample_submission.csv")
TEST_DIR     = os.path.join(BASE, "test")
TEST_GLOB    = os.path.join(TEST_DIR, "TEST_*.csv")
OUT_PATH     = os.path.join(BASE, "submission.csv")

# ======================
# Holidays
# ======================
kr_holidays = holidays.KR(years=[2023, 2024, 2025])
holiday_df = pd.DataFrame(list(dict(kr_holidays.items()).items()), columns=['date', 'holiday_name'])

# ======================
# 1) Domain features
# ======================
def add_domain_features(df, holiday_df, date_col='date'):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.normalize()

    # holiday_df 정리
    hd = holiday_df.copy()
    if 'date' not in hd.columns:
        for c in hd.columns:
            if ('date' in str(c).lower()) or ('일자' in str(c)):
                hd = hd.rename(columns={c: 'date'})
                break
        else:
            raise ValueError("holiday_df에 날짜 컬럼이 없습니다.")
    hd['date'] = pd.to_datetime(hd['date']).dt.normalize()

    hol = hd[['date']].drop_duplicates().assign(is_holiday=1)
    df = df.merge(hol, on='date', how='left')
    df['is_holiday'] = df['is_holiday'].fillna(0).astype('int8')

    w = df[date_col].dt.weekday
    df['weekday']    = w.astype('int8')
    df['is_weekend'] = w.isin([5,6]).astype('int8')
    df['month']      = df[date_col].dt.month.astype('int8')
    df['quarter']    = df[date_col].dt.quarter.astype('int8')
    df['season']     = ((df['month'] % 12)//3 + 1).astype('int8')
    df['day']        = df[date_col].dt.day.astype('int8')

    # 휴일까지 거리
    hol_days = np.sort(hd['date'].to_numpy(dtype='datetime64[D]'))
    d_days   = df[date_col].to_numpy(dtype='datetime64[D]')
    idx_next = np.searchsorted(hol_days, d_days, side='left')
    idx_prev = np.searchsorted(hol_days, d_days, side='right') - 1

    days_to_next    = np.full(len(df), np.nan)
    days_since_last = np.full(len(df), np.nan)

    m_next = idx_next < len(hol_days)
    days_to_next[m_next] = (hol_days[idx_next[m_next]] - d_days[m_next]).astype('timedelta64[D]').astype(int)

    m_prev = idx_prev >= 0
    days_since_last[m_prev] = (d_days[m_prev] - hol_days[idx_prev[m_prev]]).astype('timedelta64[D]').astype(int)

    df['days_to_next_holiday']    = pd.Series(days_to_next).astype('Int16')
    df['days_since_last_holiday'] = pd.Series(days_since_last).astype('Int16')
    m_hol = df['is_holiday'] == 1
    df.loc[m_hol, ['days_to_next_holiday', 'days_since_last_holiday']] = 0
    return df

# ======================
# 2) 윈도우 생성
# ======================
def make_windows_fast(df, group_col='store_menu', date_col='date',
                      target_col='sales_count', feature_cols=None,
                      lookback=28, horizon=7, fill_distance=999):

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    for c in ['days_to_next_holiday','days_since_last_holiday']:
        if c in df.columns: df[c] = df[c].fillna(fill_distance)

    if feature_cols is None:
        base = ['is_holiday','weekday','is_weekend','month','quarter','season','day',
                'days_to_next_holiday','days_since_last_holiday']
        feature_cols = ['sales_count'] + [c for c in base if c in df.columns]

    X_parts, Y_parts, META = [], [], []
    for k, g in df.groupby(group_col, sort=False):
        g = g.sort_values(date_col).reset_index(drop=True)

        cols_need = [date_col] + feature_cols + [target_col]
        cols_need = list(dict.fromkeys(cols_need))
        if any(c not in g.columns for c in cols_need):
            continue
        g = g[cols_need]

        T = len(g); n = T - lookback - horizon + 1
        if n <= 0:
            continue

        arr = g[feature_cols].to_numpy(np.float32)
        tgt = g[target_col].to_numpy(np.float32)
        starts = np.arange(n, dtype=np.int32)

        in_idx  = starts[:, None] + np.arange(lookback)
        out_idx = starts[:, None] + lookback + np.arange(horizon)

        X_parts.append(arr[in_idx])       # (n,28,F)
        Y_parts.append(tgt[out_idx])      # (n,7)

        META.extend({
            'group': k,
            'anchor_date': g.loc[s+lookback-1, date_col],
            'horizon_start': g.loc[s+lookback, date_col],
            'horizon_end': g.loc[s+lookback+horizon-1, date_col],
        } for s in starts)

    X_seq = np.concatenate(X_parts, 0) if X_parts else np.empty((0, lookback, len(feature_cols)), np.float32)
    y_seq = np.concatenate(Y_parts, 0) if Y_parts else np.empty((0, horizon), np.float32)
    return X_seq, y_seq, META, feature_cols

def flatten_for_tree(X_3d):
    n, L, F = X_3d.shape
    return X_3d.reshape(n, L*F)

def make_window_level_features(X_seq, used_feats):
    N, L, F = X_seq.shape
    idx_sc = used_feats.index('sales_count')
    sc = X_seq[:, :, idx_sc].astype(float)

    def lag_k(k): return sc[:, -k] if L >= k else np.zeros(N)
    def tail_mean(k): k=min(k,L); return sc[:, -k:].mean(1)
    def tail_std(k):  k=min(k,L); return sc[:, -k:].std(1)

    lag1, lag3, lag7, lag14, lag28 = lag_k(1), lag_k(3), lag_k(7), lag_k(14), lag_k(28)
    r7_mean, r7_std = tail_mean(7), tail_std(7)
    r14_mean, r14_std = tail_mean(14), tail_std(14)
    r28_mean, r28_std = tail_mean(28), tail_std(28)

    t = np.arange(L, dtype=float)
    t = (t - t.mean()) / (t.std() + 1e-8)
    slope = ((sc * t).mean(1) - sc.mean(1) * t.mean()) / (t.var() + 1e-8)

    feats = [lag1, lag3, lag7, lag14, lag28,
             r7_mean, r14_mean, r28_mean, r7_std, r14_std, r28_std, slope]

    if 'weekday' in used_feats:
        wd = X_seq[:, :, used_feats.index('weekday')]
        anchor = wd[:, -1][:, None]
        m = (wd == anchor)
        same_wd_mean = (sc * m).sum(1) / np.maximum(m.sum(1), 1)
        feats.append(same_wd_mean)

    if 'is_weekend' in used_feats:
        we = X_seq[:, :, used_feats.index('is_weekend')]
    elif 'weekday' in used_feats:
        wd = X_seq[:, :, used_feats.index('weekday')]
        we = (wd >= 5).astype(float)
    else:
        we = np.zeros_like(sc)

    weekend_mean = (sc * we).sum(1) / np.maximum(we.sum(1), 1)
    weekday_mean = (sc * (1 - we)).sum(1) / np.maximum((1 - we).sum(1), 1)
    feats += [weekend_mean, weekday_mean]

    return np.column_stack(feats)

# ======================
# 3) 학습
# ======================
def smape_ignore_zero(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    m = y_true != 0
    if m.sum()==0: return np.nan
    num = np.abs(y_pred[m]-y_true[m]); den = (np.abs(y_true[m])+np.abs(y_pred[m]))/2
    return np.mean(num/np.maximum(den,1e-8))*100

def iter_time_folds(order, n_splits=5):
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for tr, va in tscv.split(order):
        yield order[tr], order[va]

def feval_smape_plain(y_true, y_pred):
    return ("smape", smape_ignore_zero(y_true, np.clip(y_pred,0,None)), False)

def make_feval_smape_log(shift):
    def _f(y_true, y_pred):
        y_hat = np.expm1(y_pred) - shift
        y_hat = np.clip(y_hat, 0, None)
        return ("smape", smape_ignore_zero(y_true, y_hat), False)
    return _f

# ----- Load & enrich
train = pd.read_csv(TRAIN_PATH, parse_dates=['영업일자'])
train = train.rename(columns={'영업일자':'date','영업장명_메뉴명':'store_menu','매출수량':'sales_count'})
train_enriched = add_domain_features(train, holiday_df, date_col='date')

# ----- Build windows
X_seq, y_seq, meta, used_feats = make_windows_fast(train_enriched)
X_flat = flatten_for_tree(X_seq)
X_win  = make_window_level_features(X_seq, used_feats)
X_aug  = np.hstack([X_flat, X_win])

anchor = np.array([m['anchor_date'] for m in meta])
order = np.argsort(anchor)

def make_sample_weights(meta, upweight=1.0):
    w = np.ones(len(meta), float)
    if upweight!=1.0:
        for i,m in enumerate(meta):
            if str(m['group']).split('_')[0] in ('담하','미라시아'):
                w[i] = upweight
    return w

sample_weight = make_sample_weights(meta, upweight=1.0)

use_log = True; es_rounds = 100
H = y_seq.shape[1]
models = []; cv_scores = []

for h in range(H):
    y = y_seq[:, h].astype(float)
    shift = max(1e-6, -float(np.min(y)) + 1e-6) if use_log else 0.0

    fold_scores, fold_models = [], []
    for tr_idx, va_idx in iter_time_folds(order):
        X_tr, X_va = X_aug[tr_idx], X_aug[va_idx]
        y_tr_raw, y_va_raw = y[tr_idx], y[va_idx]
        w_tr = sample_weight[tr_idx]

        if use_log:
            y_tr = np.log1p(y_tr_raw + shift)
            eval_label = y_va_raw
            feval = make_feval_smape_log(shift)
        else:
            y_tr = y_tr_raw
            eval_label = y_va_raw
            feval = feval_smape_plain

        reg = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=1200, learning_rate=0.05,
            num_leaves=127, max_depth=-1, min_child_samples=20,
            subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.1, reg_lambda=0.1,
            random_state=42, n_jobs=-1
        )
        reg.fit(X_tr, y_tr, sample_weight=w_tr,
                eval_set=[(X_va, eval_label)], eval_metric=feval,
                callbacks=[lgb.early_stopping(stopping_rounds=es_rounds, verbose=False)])

        y_hat = reg.predict(X_va)
        if use_log: y_hat = np.expm1(y_hat) - shift
        y_hat = np.clip(y_hat, 0, None)

        sc = smape_ignore_zero(y_va_raw, y_hat)
        if np.isfinite(sc):
            fold_scores.append(sc); fold_models.append(reg)

    if len(fold_scores)==0:
        models.append(None); print(f"H+{h+1} CV SMAPE: N/A")
    else:
        mean_sc = float(np.mean(fold_scores))
        best_i = int(np.argmin(fold_scores))
        best_md = fold_models[best_i]
        models.append((best_md, shift, use_log))
        best_iter = getattr(best_md, "best_iteration_", None)
        print(f"H+{h+1} CV SMAPE (mean): {mean_sc:.3f}% | Best: {fold_scores[best_i]:.3f}%"
              + (f" (best_iter={best_iter})" if best_iter is not None else ""))
        cv_scores.append(mean_sc)

overall = float(np.mean(cv_scores)) if cv_scores else np.nan
print("\n=== Overall CV SMAPE ===")
print(f"{overall:.3f}%" if np.isfinite(overall) else "N/A")

# ----- 최종 재학습
final_models = []
for h in range(H):
    cv_model, shift, use_log = models[h]
    y = y_seq[:, h].astype(float)
    best_iter = getattr(cv_model, "best_iteration_", None) or cv_model.get_params().get("n_estimators", 800)
    params = cv_model.get_params(); params.update(dict(n_estimators=best_iter, objective="regression", n_jobs=-1))
    mdl = lgb.LGBMRegressor(**params)
    y_train = np.log1p(y + shift) if use_log else y
    mdl.fit(X_aug, y_train, sample_weight=sample_weight)
    final_models.append((mdl, shift, use_log))

def predict_7(models_with_meta, X_input):
    preds = []
    for mdl, shift, use_log in models_with_meta:
        p = mdl.predict(X_input)
        if use_log:
            p = np.expm1(p) - shift
        preds.append(np.clip(p, 0, None))
    return np.column_stack(preds)

# ======================
# 4) 테스트 시퀀스 빌더 (개선: 마지막 28일 사용 + META 풍부화)
# ======================
def _detect_date_col(df):
    for c in ['date','영업일자','일자','dt']:
        if c in df.columns: return c
    for c in df.columns:
        lc = str(c).lower()
        if ('date' in lc) or ('일자' in c) or ('영업일' in c):
            return c
    raise KeyError("날짜 컬럼 못 찾음")

def _detect_group_col(df):
    if 'store_menu' in df.columns: return 'store_menu'
    if '영업장명_메뉴명' in df.columns: return '영업장명_메뉴명'
    for c in df.columns:
        if '메뉴' in str(c) and '영업장' in str(c):
            return c
    raise KeyError("그룹 컬럼 못 찾음")

def _detect_sales_col(df):
    if '매출수량' in df.columns: return '매출수량'
    for a in ['sales_count','sales','qty','quantity','count','판매수량','판매수','수량','매출수량','매출건수','건수']:
        if a in df.columns: return a
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if num_cols:
        return pd.Series({c: df[c].var() for c in num_cols}).idxmax()
    raise KeyError("판매량 컬럼 못 찾음")

def build_test_sequences_flex(df_test, holiday_df, used_feats):
    date_col  = _detect_date_col(df_test)
    group_col = _detect_group_col(df_test)

    df = add_domain_features(df_test, holiday_df, date_col=date_col)
    df = df.sort_values([group_col, date_col]).reset_index(drop=True)

    X_list, META = [], []
    for g, sub in df.groupby(group_col, sort=False):
        sub = sub.sort_values(date_col).reset_index(drop=True)
        sub = sub.tail(28).reset_index(drop=True)  # 마지막 28일
        if len(sub) < 14:
            continue

        sales_col = _detect_sales_col(sub) if 'sales_count' not in sub.columns else 'sales_count'
        missing_others = [c for c in used_feats if c != 'sales_count' and c not in sub.columns]
        if missing_others:
            print(f"[WARN] {g}: missing {missing_others} → skip")
            continue

        cols   = [sales_col if c == 'sales_count' else c for c in used_feats]
        X_28F  = sub[cols].to_numpy(np.float32)
        X_flat = X_28F.reshape(1, -1)
        X_win  = make_window_level_features(X_28F[None, ...], used_feats)
        X_aug  = np.hstack([X_flat, X_win])

        last28_sales   = sub[sales_col].to_numpy(np.float32)
        last28_weekday = (sub['weekday'].to_numpy(np.int32)
                          if 'weekday' in sub.columns else
                          pd.to_datetime(sub[date_col]).dt.weekday.to_numpy(np.int32))
        anchor     = pd.to_datetime(sub[date_col].iloc[-1])
        pred_dates = [anchor + pd.Timedelta(days=d) for d in range(1,8)]
        store_name = str(g).split('_')[0]

        META.append({
            "group": g,
            "store_name": store_name,
            "anchor_date": anchor,
            "pred_dates": pred_dates,
            "mean7":  float(last28_sales[-7:].mean()) if len(last28_sales)>=7 else float(last28_sales.mean()),
            "mean28": float(last28_sales.mean()),
            "last28_sales": last28_sales,
            "last28_weekday": last28_weekday
        })
        X_list.append(X_aug)

    if not X_list:
        return np.empty((0,0), np.float32), []
    return np.vstack(X_list), META

# ======================
# 5) Post-process: 적응형 ε + 요일 naive 백업
# ======================
WINTER_STORES   = ["느티나무", "미라시아", "화담숲주막", "화담숲카페", "연회장"]
WEEKEND_STORES  = ["라그로타", "포레스트릿", "카페테리아"]
EPS_BASE        = 0.08
EPS_WINTER      = 0.20
EPS_WEEKEND     = 0.20

def _is_hot_store(store_name: str):
    s = str(store_name); return any(x in s for x in WEEKEND_STORES)

def _is_cold_store(store_name: str):
    s = str(store_name); return any(x in s for x in WINTER_STORES)

def _build_dow_naive_from_last28(last28_sales: np.ndarray, last28_weekday: np.ndarray, pred_dates: list):
    m = {}
    for d in range(7):
        vals = last28_sales[last28_weekday == d]
        m[d] = float(vals.mean()) if vals.size>0 else float(last28_sales.mean())
    return np.array([m[pd.to_datetime(dt).weekday()] for dt in pred_dates], dtype=float)

def _compute_adaptive_eps(store_name, mean7, mean28, pred_dates):
    eps = EPS_BASE + 0.09*max(mean7,0.0) + 0.03*max(mean28,0.0)
    if any(pd.to_datetime(d).month in (12,1,2) for d in pred_dates) and _is_cold_store(store_name):
        eps = max(eps, EPS_WINTER)
    if any(pd.to_datetime(d).weekday()>=5 for d in pred_dates) and _is_hot_store(store_name):
        eps = max(eps, EPS_WEEKEND)
    cap = 0.40*max(mean7, 0.0)
    if cap > 0:
        eps = min(eps, cap)
    return max(eps, 0.03)

# ======================
# 6) 제출 생성: pred ⇒ naive ⇒ menu_mean ⇒ global
# ======================
sample = pd.read_csv(SAMPLE_PATH)
submit = sample.copy()
submit.iloc[:, 1:] = 0.0

pred_dict  = {}
naive_dict = {}

test_files = sorted(glob.glob(TEST_GLOB))
if not test_files:
    raise FileNotFoundError(f"테스트 파일이 없습니다: {TEST_GLOB}")

for fpath in test_files:
    fname = os.path.basename(fpath)
    test_id = os.path.splitext(fname)[0]

    df_test = pd.read_csv(fpath)
    if 'date' not in df_test.columns:
        dc = _detect_date_col(df_test)
        df_test[dc] = pd.to_datetime(df_test[dc])
        df_test = df_test.rename(columns={dc: 'date'})
    if 'store_menu' not in df_test.columns:
        gc = _detect_group_col(df_test)
        df_test = df_test.rename(columns={gc: 'store_menu'})

    X_aug_t, META = build_test_sequences_flex(df_test, holiday_df, used_feats)
    if len(META) == 0 or X_aug_t.shape[0] == 0:
        print(f"[WARN] {fname}: usable groups=0 → skip")
        continue

    Y_hat = predict_7(final_models, X_aug_t)

    # ---- 적응형 ε + 요일 naive 10% 블렌딩 ----
    Y_hat_eps = Y_hat.copy()
    for i, m in enumerate(META):
        store_name = m.get('store_name', str(m.get('group','')).split('_')[0])
        pred_dates = m.get('pred_dates') or [pd.to_datetime(m.get('anchor_date')) + pd.Timedelta(days=d) for d in range(1,8)]
        mean7  = float(m.get('mean7', 0.0))
        mean28 = float(m.get('mean28', mean7))

        eps = _compute_adaptive_eps(store_name, mean7, mean28, pred_dates)
        Y_hat_eps[i, :] = np.where(Y_hat_eps[i, :] < eps, eps, Y_hat_eps[i, :])

        if ('last28_sales' in m) and ('last28_weekday' in m):
            naive7 = _build_dow_naive_from_last28(
                np.asarray(m['last28_sales'], dtype=float),
                np.asarray(m['last28_weekday'], dtype=int),
                pred_dates
            )
            naive7 = np.maximum(naive7, eps)
            Y_hat_eps[i, :] = 0.90 * Y_hat_eps[i, :] + 0.10 * naive7

        col = m["group"]
        for d, val in enumerate(Y_hat_eps[i], start=1):
            row_key = f"{test_id}+{d}일"
            pred_dict[(row_key, col)] = float(val)

        if ('last28_sales' in m) and ('last28_weekday' in m):
            naive7 = _build_dow_naive_from_last28(
                np.asarray(m['last28_sales'], dtype=float),
                np.asarray(m['last28_weekday'], dtype=int),
                pred_dates
            )
            for d, nv in enumerate(naive7, start=1):
                row_key = f"{test_id}+{d}일"
                naive_dict[(row_key, col)] = float(max(nv, EPS_BASE))

# ----- 백업 통계 (menu mean / global)
menu_mean = train_enriched.groupby('store_menu')['sales_count'].mean()
gmean     = float(train_enriched['sales_count'].mean())

# ----- 최종 테이블 채우기
filled = miss_pred = miss_naive = miss_menu = 0
for i in submit.index:
    row_key = submit.iloc[i, 0]
    for col in submit.columns[1:]:
        v = pred_dict.get((row_key, col))
        if v is None:
            miss_pred += 1
            v = naive_dict.get((row_key, col))
            if v is None:
                miss_naive += 1
                v = float(menu_mean.get(col, np.nan))
                if not np.isfinite(v):
                    miss_menu += 1
                    v = gmean
            v = max(v, EPS_BASE)
        submit.iat[i, submit.columns.get_loc(col)] = float(v)

print(f"[fill] pred={filled}, miss_pred={miss_pred}, miss_naive={miss_naive}, miss_menu={miss_menu}")

# ----- 안전 처리
arr = submit.iloc[:,1:].to_numpy(float)
arr = np.nan_to_num(arr, nan=gmean, posinf=gmean, neginf=0.0)
arr = np.maximum(arr, EPS_BASE)
submit.iloc[:,1:] = arr

submit.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {OUT_PATH}")